# Baseline check notebook -- pytorch image

Simulates a normal user session: install a package via pip, run computational
code against it (using pandas, already provided by the scipy-notebook base
image), clean up the pip installs, then validate that cleanup actually
restored the environment to its pre-check state.

Run with: `jupyter nbconvert --to notebook --execute baseline_check.ipynb`.
A failed assertion in any cell will make nbconvert exit non-zero.

In [ ]:
import subprocess
import sys


def installed_packages():
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True, text=True, check=True,
    )
    return set(result.stdout.splitlines())


def pip_install(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *packages],
        check=True,
    )


def pip_uninstall(*packages):
    if packages:
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", *packages],
            check=True,
        )


def fresh_process_can_import(module_name):
    result = subprocess.run([sys.executable, "-c", f"import {module_name}"], capture_output=True)
    return result.returncode == 0


baseline_packages = installed_packages()
print(f"Baseline package count: {len(baseline_packages)}")

## Install packages

In [ ]:
pip_install("tabulate")

after_install_packages = installed_packages()
newly_installed = sorted(after_install_packages - baseline_packages)
print("Newly installed packages:", newly_installed)

assert newly_installed, "Expected pip install to add at least one new package"

## Run computational code using pandas (pre-installed) and the newly installed tabulate

In [ ]:
import pandas as pd
from tabulate import tabulate

df = pd.DataFrame({
    "region": ["north", "north", "south", "south", "east"],
    "sales": [120, 150, 90, 110, 200],
    "units": [10, 12, 8, 9, 15],
})

summary = df.groupby("region").agg(total_sales=("sales", "sum"), total_units=("units", "sum"))
summary["avg_price"] = summary["total_sales"] / summary["total_units"]

print(tabulate(summary, headers="keys"))

assert summary.loc["north", "total_sales"] == 270
assert summary.loc["south", "total_units"] == 17
assert round(summary.loc["east", "avg_price"], 4) == round(200 / 15, 4)
print("PASS: pandas computation produced expected results")

## Clean up the pip installs from earlier

In [ ]:
package_names = [pkg.split("==")[0] for pkg in newly_installed]
pip_uninstall(*package_names)
print("Uninstalled:", package_names)

## Validate cleanup was successful

Checks re-importability of whatever was actually newly installed
(`package_names`), not a fixed package name -- pandas is part of the
scipy-notebook base image this Dockerfile builds on, so it's correctly never
uninstalled and should stay importable; only the packages we actually
installed and removed (e.g. tabulate) should become unimportable.

Import checks run in a fresh subprocess rather than this kernel, since this
kernel already has `tabulate` cached in `sys.modules` from the cell above.

In [ ]:
after_cleanup_packages = installed_packages()
leftover = after_cleanup_packages - baseline_packages
missing = baseline_packages - after_cleanup_packages
still_importable = {name: fresh_process_can_import(name) for name in package_names}

print("Leftover packages after cleanup:", sorted(leftover))
print("Packages missing that were present at baseline:", sorted(missing))
for name, importable in sorted(still_importable.items()):
    print(f"{name} importable in a fresh process after cleanup:", importable)

ok = not leftover and not missing and not any(still_importable.values())
if ok:
    print("PASS: environment restored to baseline after cleanup")
else:
    raise AssertionError("FAIL: environment was not cleanly restored to baseline")